# Notebook 1: Preprocessing & Segmentation

This notebook walks you through the first stage of the `liv_zones` pipeline: loading a raw microscopy image, running segmentation to identify cells and organelles, and computing distance maps that encode each object's position within the hepatic acinus.

By the end of this notebook you will have produced:
- **Instance segmentation masks** for cells, mitochondria, lipid droplets, and peroxisomes
- **Distance maps** from the central vein, portal vein, and cell boundaries

These files are the inputs for Notebook 2 (Feature Extraction).

---

## Biological background

The liver is divided into repeating functional units called **hepatic acini**. Each acinus spans the distance between a **portal vein** (bringing nutrient-rich blood from the gut) and a **central vein** (draining blood toward the heart). Hepatocytes (liver cells) along this axis are exposed to different oxygen and nutrient concentrations, causing them to specialize — a phenomenon called **liver zonation**.

`liv_zones` quantifies this zonation by measuring organelle properties at precise positions along the portal-to-central axis.

---

> **GPU note:** Segmentation (Section 4) requires a CUDA-capable GPU and the pretrained Cellpose models. If you do not have a GPU available, **skip Section 4** and use the pre-computed masks provided in `sample_data/output/` to continue with the rest of this notebook and with Notebook 2.

## 1. Setup

Import the packages we need and define file paths. Adjust `IMAGE_PATH` and `SAVE_PATH` to point at your own data when working with real images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from cellpose.io import imread

from liv_zones.preprocess import preprocessing

# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust these to point at your image and desired output folder.
IMAGE_PATH = "sample_data/sample_image.tif"
SAVE_PATH  = "sample_data/output"

# Create the output directory if it doesn't exist
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)

# ── Scale ──────────────────────────────────────────────────────────────────
# The number of pixels per micron in your image.
# To find this value in FIJI: Image > Properties, then read "pixels per micron"
# or measure the scale bar.
SCALE = 14.4024  # pixels per micron (typical for 40x confocal)

print(f"Image:  {IMAGE_PATH}")
print(f"Output: {SAVE_PATH}")
print(f"Scale:  {SCALE} px/µm")

## 2. Understanding the input image

The input is a **multi-channel TIF** — a single file that stores several fluorescence channels stacked together. Each channel was acquired with a different dye that highlights a specific cellular structure:

| Channel index | Structure | Why we image it |
|:---:|:---|:---|
| 0 | **Actin** | Outlines the cell boundary — used to segment individual cells |
| 1 | **Mitochondria** | Energy-producing organelles; shape changes with metabolic state |
| 2 | **Lipid droplets** | Fat-storage organelles; accumulate in disease |
| 3 | **Peroxisomes** | Detoxification organelles; small punctate structures |

Let's load the image and display each channel side by side.

In [ ]:
image = imread(IMAGE_PATH)
print(f"Image shape: {image.shape}  →  (channels, height, width)")
print(f"Data type:   {image.dtype}")

In [ ]:
channel_names = ["Actin (ch 0)", "Mitochondria (ch 1)",
                 "Lipid droplets (ch 2)", "Peroxisomes (ch 3)"]
cmaps = ["Greens", "Reds", "Blues", "Oranges"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, name, cmap) in enumerate(zip(axes, channel_names, cmaps)):
    ch = image[i]
    # Clip to the 1st–99th percentile to improve contrast
    vmin, vmax = np.percentile(ch, [1, 99])
    ax.imshow(ch, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle("Raw image — one channel per panel", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. Running preprocessing

The `preprocessing()` function does three things in sequence:

1. **Segments each organelle** using pretrained Cellpose models. It outputs a labeled *instance segmentation mask* — an array of integers where each unique nonzero value identifies one distinct organelle.
2. **Computes vein distance maps** using the cell mask geometry to find the two vein regions, then runs a Euclidean distance transform from each vein.
3. **Computes the cell-boundary distance map** — how far each pixel is from the nearest cell boundary.

### Arguments

| Argument | Description |
|:---|:---|
| `image_path` | Path to the multi-channel TIF |
| `save_path` | Directory where all output `.npy` files are written |
| `channels` | Dict mapping channel name → channel index in the TIF |
| `feature_list` | List of features to compute. If `None`, automatically detects what's missing. |

### Channel dictionary

The `channels` dict tells `preprocessing()` which TIF channel to pass to each segmentation model.

> **Important:** If your channel order is different, update the integers below to match.

In [ ]:
# Map channel names to their integer indices in the TIF
channels = {
    "actin":  0,   # used to segment cell boundaries
    "mito":   1,   # mitochondria channel
    "lipid":  2,   # lipid droplet channel
    "peroxi": 3,   # peroxisome channel
}

# Which features to compute.
# Comment out any that you don't need to (re-)compute.
feature_list = [
    "cell_mask.npy",
    "mito_mask.npy",
    "lipid_droplet_mask.npy",
    "peroxisome_mask.npy",
    "central_dist.npy",
    "portal_dist.npy",
    "boundary_dist.npy",
]

In [ ]:
# ⚠️  This cell requires a CUDA GPU.  Skip it if you don't have one.
# Pre-computed masks are available in sample_data/output/

preprocessing(
    image_path=IMAGE_PATH,
    save_path=SAVE_PATH,
    channels=channels,
    feature_list=feature_list,
)

print("Preprocessing complete!")

### What was written to disk?

After running `preprocessing()`, the output directory should contain:

In [ ]:
import os

output_files = sorted(os.listdir(SAVE_PATH))
print(f"Files in {SAVE_PATH}:")
for f in output_files:
    fpath = os.path.join(SAVE_PATH, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:<35}  {size_kb:>8.1f} KB")

## 4. Inspecting segmentation results

An **instance segmentation mask** is a 2D integer array where:
- `0` = background (no object)
- `1, 2, 3, ...` = unique IDs for individual objects

This means you can count organelles simply by checking `mask.max()` (the highest ID = number of objects), or isolate a single organelle with `mask == 5`.

Let's load all four masks and visualise them overlaid on the raw image.

In [ ]:
cell_mask  = np.load(f"{SAVE_PATH}/cell_mask.npy")
mito_mask  = np.load(f"{SAVE_PATH}/mito_mask.npy")
lipid_mask = np.load(f"{SAVE_PATH}/lipid_droplet_mask.npy")
perox_mask = np.load(f"{SAVE_PATH}/peroxisome_mask.npy")

print("Mask shapes (height × width):")
print(f"  cell_mask:   {cell_mask.shape}  → {cell_mask.max()} cells")
print(f"  mito_mask:   {mito_mask.shape}  → {mito_mask.max()} mitochondria")
print(f"  lipid_mask:  {lipid_mask.shape}  → {lipid_mask.max()} lipid droplets")
print(f"  perox_mask:  {perox_mask.shape}  → {perox_mask.max()} peroxisomes")

In [ ]:
from cellpose import utils as cp_utils

def show_mask_overlay(raw_channel, mask, title, ax, raw_cmap="gray"):
    """Display a raw channel with mask outlines overlaid."""
    vmin, vmax = np.percentile(raw_channel, [1, 99])
    ax.imshow(raw_channel, cmap=raw_cmap, vmin=vmin, vmax=vmax)

    # Draw outlines of segmented objects in red
    outlines = cp_utils.masks_to_outlines(mask)
    overlay = np.zeros((*outlines.shape, 4), dtype=float)  # RGBA
    overlay[outlines == 1] = [1, 0.2, 0.2, 0.8]           # red outlines
    ax.imshow(overlay)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_mask_overlay(image[0], cell_mask,  "Cell mask\n(actin channel)",        axes[0], "Greens")
show_mask_overlay(image[1], mito_mask,  "Mito mask\n(mito channel)",          axes[1], "Reds")
show_mask_overlay(image[2], lipid_mask, "Lipid droplet mask\n(lipid channel)", axes[2], "Blues")
show_mask_overlay(image[3], perox_mask, "Peroxisome mask\n(perox channel)",   axes[3], "Oranges")

plt.suptitle("Segmentation masks (red outlines) overlaid on raw image", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Inspecting distance maps

### Vein distance maps

Two distance maps are computed from the vein geometry detected in the cell mask:

- `central_dist.npy` — distance (in pixels) of every pixel from the **central vein**
- `portal_dist.npy`  — distance (in pixels) of every pixel from the **portal vein**

These are used to compute the **acinus position** of each organelle:

$$\text{acinus position} = \frac{d_{\text{portal}} - d_{\text{central}}}{d_{\text{portal}} + d_{\text{central}}}$$

This normalises position to the range **[−1, +1]**, where:
- **−1** = at the central vein
- **0**  = midpoint between veins
- **+1** = at the portal vein

### Cell-boundary distance map

`boundary_dist.npy` stores, for each pixel, its distance (in pixels) to the nearest cell boundary. This is used to compute each organelle's relative position within the cell (near the edge vs. near the centre).

In [ ]:
central_dist  = np.load(f"{SAVE_PATH}/central_dist.npy")
portal_dist   = np.load(f"{SAVE_PATH}/portal_dist.npy")
boundary_dist = np.load(f"{SAVE_PATH}/boundary_dist.npy")

# Compute acinus position map for visualisation
with np.errstate(invalid="ignore"):
    acinus_pos = (portal_dist - central_dist) / (portal_dist + central_dist)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(central_dist, cmap="plasma")
axes[0].set_title("Distance from central vein (px)", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(portal_dist, cmap="plasma")
axes[1].set_title("Distance from portal vein (px)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(acinus_pos, cmap="RdBu", vmin=-1, vmax=1)
axes[2].set_title("Acinus position\n(−1 = central vein, +1 = portal vein)", fontsize=10)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle("Vein distance maps", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(boundary_dist, cmap="viridis")
ax.set_title("Distance from nearest cell boundary (px)", fontsize=10)
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Summary

You have completed the preprocessing stage. The following files are now in `sample_data/output/`:

| File | Contents |
|:---|:---|
| `cell_mask.npy` | Instance segmentation of cells |
| `mito_mask.npy` | Instance segmentation of mitochondria |
| `lipid_droplet_mask.npy` | Instance segmentation of lipid droplets |
| `peroxisome_mask.npy` | Instance segmentation of peroxisomes |
| `central_dist.npy` | Distance transform from central vein |
| `portal_dist.npy` | Distance transform from portal vein |
| `boundary_dist.npy` | Distance transform from cell boundaries |

**Next:** Open `02_feature_extraction.ipynb` to extract quantitative features from these masks.